# 🧪 HUẤN LUYỆN FACT-CHECKER AGENT (SCIBERT / DEBERTA NLI)
### Dự án: Multi-Agent Research RAG System
> **Môi trường chạy:** Kaggle Notebook (GPU T4 x2 hoặc 1 GPU T4 16GB)
> **Mục tiêu:** Huấn luyện mô hình NLI 3 nhãn (SUPPORT, CONTRADICT, NOT_ENOUGH_INFO) để kiểm chứng factual grounding cho các bài báo khoa học.

## Bước 1: Cài đặt thư viện cần thiết

In [ ]:
!pip install -q -U transformers datasets accelerate scikit-learn torch

import os
import torch

# Khóa cố định GPU 0 để tránh phân mảnh bộ nhớ giữa 2 GPU T4
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## Bước 2: Tải và Chuẩn bị Dữ liệu SciFact (AllenAI) + Tập AI Domain

In [ ]:
import json
import urllib.request
from pathlib import Path
from collections import Counter

DATA_DIR = Path("./scifact_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

URLS = {
    "corpus": "https://scifact.s3-us-west-2.amazonaws.com/data/corpus.jsonl",
    "claims_train": "https://scifact.s3-us-west-2.amazonaws.com/data/claims_train.jsonl",
    "claims_dev": "https://scifact.s3-us-west-2.amazonaws.com/data/claims_dev.jsonl",
}

for name, url in URLS.items():
    fpath = DATA_DIR / f"{name}.jsonl"
    if not fpath.exists() or fpath.stat().st_size == 0:
        print(f"Đang tải {name}.jsonl...")
        urllib.request.urlretrieve(url, fpath)
        print(f"-> Đã tải xong {name}.jsonl ({fpath.stat().st_size} bytes)")
    else:
        print(f"{name}.jsonl đã có sẵn ({fpath.stat().st_size} bytes).")

# Đọc Corpus
corpus = {}
with open(DATA_DIR / "corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            item = json.loads(line)
            corpus[str(item["doc_id"])] = {
                "title": item.get("title", ""),
                "sentences": item.get("abstract", [])
            }
print(f"Tổng số tài liệu trong Corpus: {len(corpus)}")

## Bước 3: Ghép cặp NLI (Premise, Hypothesis, Label)

In [ ]:
LABEL2ID = {"CONTRADICT": 0, "NOT_ENOUGH_INFO": 1, "SUPPORT": 2}
ID2LABEL = {0: "CONTRADICT", 1: "NOT_ENOUGH_INFO", 2: "SUPPORT"}

def build_nli_pairs(claims_file: Path, corpus: dict) -> list:
    pairs = []
    with open(claims_file, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            item = json.loads(line)
            claim = item["claim"]
            evidence_dict = item.get("evidence", {})
            cited_doc_ids = [str(cid) for cid in item.get("cited_doc_ids", [])]
            has_evidence = False

            for doc_id, ev_list in evidence_dict.items():
                doc_id_str = str(doc_id)
                if doc_id_str not in corpus: continue
                doc_info = corpus[doc_id_str]
                doc_sentences = doc_info["sentences"]

                for ev in ev_list:
                    label_str = ev.get("label", "NOT_ENOUGH_INFO")
                    sent_indices = ev.get("sentences", [])
                    premise_sents = [doc_sentences[idx] for idx in sent_indices if idx < len(doc_sentences)]
                    if not premise_sents and doc_sentences:
                        premise_sents = doc_sentences[:2]
                    premise_text = " ".join(premise_sents).strip()
                    if premise_text:
                        pairs.append({
                            "premise": f"{doc_info['title']}. {premise_text}",
                            "hypothesis": claim,
                            "label": LABEL2ID.get(label_str, 1)
                        })
                        has_evidence = True

            if cited_doc_ids:
                for cid in cited_doc_ids:
                    if cid in corpus and str(cid) not in evidence_dict:
                        doc_info = corpus[cid]
                        sample_text = " ".join(doc_info["sentences"][:2]).strip()
                        if sample_text:
                            pairs.append({
                                "premise": f"{doc_info['title']}. {sample_text}",
                                "hypothesis": claim,
                                "label": LABEL2ID["NOT_ENOUGH_INFO"]
                            })
                            break
            elif not has_evidence:
                pairs.append({
                    "premise": "No supporting scientific evidence documented.",
                    "hypothesis": claim,
                    "label": LABEL2ID["NOT_ENOUGH_INFO"]
                })
    return pairs

train_pairs = build_nli_pairs(DATA_DIR / "claims_train.jsonl", corpus)
dev_pairs = build_nli_pairs(DATA_DIR / "claims_dev.jsonl", corpus)

# Bổ sung dữ liệu AI Domain (LoRA, Transformer, QLoRA)
ai_domain_pairs = [
    {"premise": "LoRA freezes pre-trained weights and injects trainable rank decomposition matrices into Transformer layers, reducing parameters by 10,000x.", "hypothesis": "LoRA freezes model weights and reduces trainable parameters by 10,000 times.", "label": 2},
    {"premise": "LoRA freezes pre-trained weights and injects trainable rank decomposition matrices into Transformer layers, reducing parameters by 10,000x.", "hypothesis": "LoRA increases trainable parameters by 10,000 times and causes high inference latency.", "label": 0},
    {"premise": "QLoRA backpropagates gradients through a frozen 4-bit quantized pretrained language model into Low Rank Adapters.", "hypothesis": "QLoRA enables 4-bit finetuning on a single GPU.", "label": 2},
    {"premise": "QLoRA backpropagates gradients through a frozen 4-bit quantized pretrained language model into Low Rank Adapters.", "hypothesis": "QLoRA requires full 32-bit floating point precision and 8 GPUs to train.", "label": 0},
    {"premise": "Transformer is based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.", "hypothesis": "Transformers use self-attention without recurrence.", "label": 2},
    {"premise": "Transformer is based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.", "hypothesis": "ResNet-50 achieves high top-1 accuracy on ImageNet visual classification.", "label": 1},
]

train_pairs.extend(ai_domain_pairs * 5) # Tăng trọng số AI domain

print(f"Train pairs: {len(train_pairs)}")
print(f"Dev pairs: {len(dev_pairs)}")
train_dist = Counter(p['label'] for p in train_pairs)
for lid, lname in ID2LABEL.items():
    print(f"  - {lname:<15}: {train_dist.get(lid, 0)} mẫu ({train_dist.get(lid, 0)/len(train_pairs)*100:.1f}%)")

## Bước 4: Khởi tạo Mô hình SciBERT & Tokenization

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

MODEL_NAME = "allenai/scibert_scivocab_uncased"
OUTPUT_DIR = "/kaggle/working/scibert_scifact_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(
        examples["premise"],
        examples["hypothesis"],
        truncation=True,
        max_length=256
    )

train_ds = Dataset.from_list(train_pairs).map(preprocess_function, batched=True)
dev_ds = Dataset.from_list(dev_pairs).map(preprocess_function, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID
)
print(f"Đã nạp mô hình {MODEL_NAME} thành công!")

## Bước 5: Huấn luyện với Trainer (Tối ưu hóa GPU T4)

In [ ]:
import numpy as np
from transformers import TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": round(acc, 4),
        "f1_macro": round(f1, 4),
        "precision_macro": round(precision, 4),
        "recall_macro": round(recall, 4)
    }

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,    # effective batch size = 32
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),   # Bật Tensor Cores GPU T4
    dataloader_num_workers=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,               # Tiết kiệm đĩa Kaggle
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=20,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Bắt đầu huấn luyện...")
trainer.train()

# Lưu checkpoint mô hình tốt nhất
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Đã lưu mô hình tốt nhất vào {OUTPUT_DIR}")

## Bước 6: Đánh giá Chi tiết & Ma trận Nhầm lẫn (Confusion Matrix)

In [ ]:
from transformers import pipeline
from sklearn.metrics import classification_report, confusion_matrix

classifier = pipeline("text-classification", model=OUTPUT_DIR, device=0 if torch.cuda.is_available() else -1)

y_true = []
y_pred = []

for item in dev_pairs[:200]:
    premise = item["premise"]
    hypothesis = item["hypothesis"]
    true_label = item["label"]

    res = classifier(f"{premise[:256]} </s></s> {hypothesis}")[0]
    pred_label_id = LABEL2ID.get(res["label"], 1)

    y_true.append(true_label)
    y_pred.append(pred_label_id)

print("=" * 60)
print("BÁO CÁO PHÂN LOẠI SCIENTIFIC NLI")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=["CONTRADICT", "NOT_ENOUGH_INFO", "SUPPORT"], zero_division=0))
print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

## Bước 7: Thử nghiệm Trực tiếp (Live Interactive Test)

In [ ]:
def verify_claim(evidence: str, claim: str):
    res = classifier(f"{evidence[:256]} </s></s> {claim}")[0]
    print(f"Chứng cứ: {evidence}")
    print(f"Luận điểm: {claim}")
    print(f"Phán quyết: {res['label'].upper()} (Độ tự tin: {res['score']:.2%})")
    print("-" * 50)

# Test 1: SUPPORT
verify_claim(
    evidence="LoRA freezes the pre-trained weights and injects trainable rank decomposition matrices, reducing parameters by 10,000x.",
    claim="LoRA reduces the number of trainable parameters by 10,000 times."
)

# Test 2: CONTRADICT
verify_claim(
    evidence="LoRA freezes the pre-trained weights and injects trainable rank decomposition matrices, reducing parameters by 10,000x.",
    claim="LoRA causes dramatic parameter growth and increases GPU memory requirements."
)

# Test 3: NOT_ENOUGH_INFO
verify_claim(
    evidence="Attention Is All You Need introduces the Transformer model relying entirely on self-attention.",
    claim="LoRA can be trained on a single smartphone with 4GB RAM."
)

## Bước 8: Đóng gói và Xuất file Mô hình

In [ ]:
import shutil

zip_output = "/kaggle/working/scibert_scifact_model"
shutil.make_archive(zip_output, 'zip', OUTPUT_DIR)
print(f"Đã nén xong: {zip_output}.zip")
print("Bạn có thể tải file zip này về máy từ tab Output bên phải màn hình Kaggle!")